# FSC Pre-Alignment Test Notebook

Validate the phase-correlation + Fourier-shift pre-alignment step before FSC.

This notebook includes:
- a synthetic known-shift test
- before/after FSC comparison
- optional real-map pair check

In [ ]:
import os
import sys
import glob
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib.pyplot as plt

from fsc.fsc_calculator import (
    compute_fsc,
    fsc_to_resolution,
    load_volume,
    apply_fourier_shift,
    estimate_translation_phase_correlation,
    align_volume_to_reference,
    compute_fsc_from_files,
)

%matplotlib inline

## 1. Synthetic Known-Shift Validation

In [ ]:
def make_gaussian_blob(shape=(96, 96, 96), sigma=10.0):
    z, y, x = np.indices(shape)
    cz, cy, cx = (np.array(shape) - 1.0) / 2.0
    r2 = (z - cz) ** 2 + (y - cy) ** 2 + (x - cx) ** 2
    return np.exp(-r2 / (2.0 * sigma ** 2)).astype(np.float32)

ref = make_gaussian_blob()
true_shift_to_create_moving = (2.75, -3.20, 1.40)  # (z, y, x)
moving = apply_fourier_shift(ref, true_shift_to_create_moving)

estimated_shift_to_apply, peak = estimate_translation_phase_correlation(ref, moving, subpixel=True)
aligned, meta = align_volume_to_reference(ref, moving, subpixel=True)

expected_shift_to_apply = tuple(-np.array(true_shift_to_create_moving))
shift_error = np.array(estimated_shift_to_apply) - np.array(expected_shift_to_apply)

print('True shift used to create moving (z,y,x):', true_shift_to_create_moving)
print('Expected corrective shift (z,y,x):      ', expected_shift_to_apply)
print('Estimated corrective shift (z,y,x):     ', estimated_shift_to_apply)
print('Shift error (vox):                       ', tuple(np.round(shift_error, 4)))
print('Phase-correlation peak:                  ', round(float(peak), 6))
print('RMS residual after alignment:            ', float(np.sqrt(np.mean((aligned - ref) ** 2))))

## 2. FSC Comparison Before vs After Alignment (Synthetic)

In [ ]:
shells_before, fsc_before = compute_fsc(ref, moving)
shells_after, fsc_after = compute_fsc(ref, aligned)

voxel_size_A = 1.0
res_before = fsc_to_resolution(shells_before, fsc_before, voxel_size_A, threshold=0.143)
res_after = fsc_to_resolution(shells_after, fsc_after, voxel_size_A, threshold=0.143)

with np.errstate(divide='ignore', invalid='ignore'):
    x_before = np.where(shells_before > 0, voxel_size_A / shells_before, np.inf)
    x_after = np.where(shells_after > 0, voxel_size_A / shells_after, np.inf)

mask_before = np.isfinite(x_before)
mask_after = np.isfinite(x_after)

fig, ax = plt.subplots(figsize=(8, 5), dpi=130)
ax.plot(x_before[mask_before], fsc_before[mask_before], label='Before alignment', linewidth=2)
ax.plot(x_after[mask_after], fsc_after[mask_after], label='After alignment', linewidth=2)
ax.axhline(0.143, color='k', linestyle='--', linewidth=1, label='FSC=0.143')
ax.set_xlim(10, 2)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Resolution (A)')
ax.set_ylabel('FSC')
ax.set_title('Synthetic FSC: Before vs After Pre-Alignment')
ax.legend()
plt.show()

print('d0.143 before alignment (A):', res_before)
print('d0.143 after alignment  (A):', res_after)

## 3. Optional Real Map Pair Check

In [ ]:
MAPS_DIR = 'maps'
mrc_files = sorted(glob.glob(f'{MAPS_DIR}/*.mrc'))
print(f'Found {len(mrc_files)} map(s)')
if len(mrc_files) >= 2:
    p1, p2 = mrc_files[0], mrc_files[1]
    print('Testing pair:')
    print('  map1:', Path(p1).name)
    print('  map2:', Path(p2).name)

    r_raw = compute_fsc_from_files(p1, p2, thresholds=[0.143, 0.5], align_map2=False)
    r_aln = compute_fsc_from_files(p1, p2, thresholds=[0.143, 0.5], align_map2=True, subpixel_alignment=True)

    print('')
    print('Without alignment:')
    print('  d0.143 (A):', r_raw['resolutions'][0.143])
    print('  d0.500 (A):', r_raw['resolutions'][0.5])

    print('With alignment:')
    print('  d0.143 (A):', r_aln['resolutions'][0.143])
    print('  d0.500 (A):', r_aln['resolutions'][0.5])
    print('  shift(z,y,x):', tuple(np.round(r_aln['alignment']['shift_zyx'], 4)))
    print('  peak corr   :', round(float(r_aln['alignment']['peak_correlation']), 6))
else:
    print('Place at least two MRC files in maps/ to run this section.')